## Format and process data

This code creates the data for each step in the forecasting process, starting with the raw weather files

### Start by importing necessary libraries

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import sys, os, json, importlib, glob
sys.path.append( os.path.abspath(os.path.join('..')) )
import utils.format_data_utils as data_utils


# Opening config file
f = open('../fpaths_config.json')
paths = json.load(f)

weather_path = paths["weather_data"]
nn_preds_path = paths["nn_abundance_predictions"]
raw_mols_path = paths["raw_mols"]

model_files_path = paths["model_files"]

## Format weather data

In [ ]:
# The ASOS data is more complete than the LCD data
#importlib.reload(data_utils)

#raw_weather = '{}/LCD_San_Juan.csv'.format(weather_path)
#data = data_utils.format_lcd(raw_weather, "San_Juan")

#data_fil = '{}/San_Juan.pd'.format(weather_path)
#data = pd.read_pickle(data_fil)
#data['Precip'] = data['Precip']/10
#data.to_pickle('{}/San_Juan_daily.pd'.format(weather_path))

In [6]:
importlib.reload(data_utils)

raw_weather = '{}/sj_asos.csv'.format(weather_path)
data = pd.read_csv(raw_weather)
data['Avg_Temp'] = (data['max_temp_f'] + data['min_temp_f']) / 2
data['Humidity'] = (data['min_rh'] + data['max_rh']) / 2
#data['Humidity'] = data['Humidity'].fillna()
data['Humidity'] = data['Humidity'].fillna(data['avg_rh'])
data.rename(columns={'precip_in': 'Precip_in', 'day': 'Datetime'}, inplace=True)
data.Datetime = pd.to_datetime(data.Datetime)

_ = data_utils.cleanDailySummaries(data, 'San_Juan')

data = pd.read_csv('{}/San_Juan.csv'.format(weather_path))
data['Precip'] = 25.4 * data['Precip_in']
data = data[['Location', 'Year', 'Month', 'Day', 'Avg_Temp', 'Precip', 'Humidity', 'Ref']]
data.to_pickle('{}/San_Juan.pd'.format(weather_path))

data['Precip'] = data['Precip']/10
data.to_pickle('{}/San_Juan_daily.pd'.format(weather_path))



c:\Users\Adrienne\Documents\Projects\Dissertation\Aedes-AI_Forecasting\utils\format_data_utils.py:238: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Datetime'] = pd.to_datetime(data['Datetime'])


In [7]:
importlib.reload(data_utils)
data_utils.collect_ceiba()

In [8]:
importlib.reload(data_utils)
raw_weather = '{}/DailySummaries_Ceiba.csv'.format(weather_path)
_ = data_utils.cleanDailySummaries(pd.read_csv(raw_weather), loc='Ceiba')

data = pd.read_csv('{}/Ceiba.csv'.format(weather_path))
data['Precip'] = 25.4 * data['Precip_in']
data = data[['Location', 'Year', 'Month', 'Day', 'Avg_Temp', 'Precip', 'Humidity', 'Ref']]
data.to_pickle('{}/Ceiba.pd'.format(weather_path))

data['Precip'] = data['Precip']/10
data.to_pickle('{}/Ceiba_daily.pd'.format(weather_path))

In [ ]:
#Didn't end up using West SJ data
#importlib.reload(data_utils)
#raw_weather = '{}/DailySummaries_westSJ.csv'.format(weather_path)
#_ = data_utils.cleanDailySummaries(raw_weather)

#data = pd.read_csv('{}/West_SJ.csv'.format(weather_path))
#data['Precip'] = 25.4 * data['Precip_in']
#data = data[['Location', 'Year', 'Month', 'Day', 'Avg_Temp', 'Precip', 'Humidity', 'Ref']]
#data.to_pickle('{}/West_SJ.pd'.format(weather_path))

#data['Precip'] = data['Precip']/10
#data.to_pickle('{}/West_SJ_daily.pd'.format(weather_path))


### Merge MoLS

Save the daily weather as csv files for MoLS

In [ ]:
importlib.reload(data_utils)
data_utils.save_weather_mols()

Obtain corresponding MoLS predictions for `loc_daily.pd` files and store them in `raw_mols_path` (manual process). Finally, 90 day burn in and burn out

In [ ]:
fils = glob.glob('{}/*_MoLS.csv'.format(raw_mols_path))
for fil in fils:
    data = pd.read_csv(fil)
    data = data.iloc[10:-90].reset_index(drop=True)
    data.to_csv(fil, index=False)